In [1]:
# Groq API: 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
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.0,
    api_key="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"  # free at console.groq.com
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000198BA096940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000198DB08E070>, model_name='qwen/qwen3-32b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
# how to get full api from its swagger?
import requests

url = "https://petstore.swagger.io/v2/swagger.json"
spec_text = requests.get(url).text
spec_text

'{"swagger":"2.0","info":{"description":"This is a sample server Petstore server.  You can find out more about Swagger at [http://swagger.io](http://swagger.io) or on [irc.freenode.net, #swagger](http://swagger.io/irc/).  For this sample, you can use the api key `special-key` to test the authorization filters.","version":"1.0.7","title":"Swagger Petstore","termsOfService":"http://swagger.io/terms/","contact":{"email":"apiteam@swagger.io"},"license":{"name":"Apache 2.0","url":"http://www.apache.org/licenses/LICENSE-2.0.html"}},"host":"petstore.swagger.io","basePath":"/v2","tags":[{"name":"pet","description":"Everything about your Pets","externalDocs":{"description":"Find out more","url":"http://swagger.io"}},{"name":"store","description":"Access to Petstore orders"},{"name":"user","description":"Operations about user","externalDocs":{"description":"Find out more about our store","url":"http://swagger.io"}}],"schemes":["https","http"],"paths":{"/pet/{petId}/uploadImage":{"post":{"tags":[

In [3]:
import json

spec_text = json.loads(spec_text)
spec_text['openapi'] = "3.1.0"
spec_text.pop('swagger')

'2.0'

In [4]:
import requests
import json

# 1. Fetch the Swagger 2.0 spec
url = "https://petstore.swagger.io/v2/swagger.json"
spec = requests.get(url).json()


# ── 2. Parse Swagger 2.0 → LLM tool definitions ──────────────────────────────

def swagger_to_tools(spec: dict) -> list[dict]:
    """Convert Swagger 2.0 spec → OpenAI-style tool definitions."""
    tools = []
    schemes = spec.get("schemes", ["https"])
    base_url = f"{schemes[0]}://{spec['host']}{spec.get('basePath', '')}"

    for path, path_item in spec["paths"].items():
        for method, operation in path_item.items():
            if method not in ("get", "post", "put", "delete", "patch"):
                continue

            properties = {}
            required_params = []

            for param in operation.get("parameters", []):
                param_in = param.get("in")

                if param_in == "body":
                    # Inline the body schema properties directly
                    body_schema = param.get("schema", {})
                    body_props = body_schema.get("properties", {})
                    properties.update(body_props)
                    required_params += body_schema.get("required", [])

                else:
                    # path / query / header params
                    prop = {
                        "type": param.get("type", "string"),
                        "description": param.get("description", f"{param_in} parameter"),
                    }
                    if "enum" in param:
                        prop["enum"] = param["enum"]

                    properties[param["name"]] = prop
                    if param.get("required"):
                        required_params.append(param["name"])

            # Sanitize operationId for use as a tool name
            op_id = operation.get(
                "operationId",
                f"{method}_{path.replace('/', '_').strip('_')}"
            )
            tool_name = op_id.replace("/", "_").replace("{", "").replace("}", "").replace("-", "_")

            tools.append({
                "_meta": {           # internal use only — not sent to LLM
                    "method": method,
                    "path": path,
                    "base_url": base_url,
                },
                "type": "function",
                "function": {
                    "name": tool_name,
                    "description": operation.get(
                        "summary",
                        operation.get("description", f"{method.upper()} {path}")
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": properties,
                        "required": required_params,
                    },
                },
            })

    return tools


tools = swagger_to_tools(spec)

# Tools ready for OpenAI / Anthropic — strip _meta before sending
llm_tools = [{"type": t["type"], "function": t["function"]} for t in tools]

print(f"Generated {len(llm_tools)} tools")
print(json.dumps(llm_tools[2], indent=2))   # preview one

Generated 20 tools
{
  "type": "function",
  "function": {
    "name": "updatePet",
    "description": "Update an existing pet",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": []
    }
  }
}


In [5]:
# ── 3. Tool executor — runs the actual API call ───────────────────────────────

def execute_tool(tool_name: str, arguments: dict, tools: list[dict]) -> dict:
    """Given a tool_name + arguments from LLM, make the HTTP request."""

    # Find the matching tool (with _meta)
    tool = next((t for t in tools if t["function"]["name"] == tool_name), None)
    if not tool:
        raise ValueError(f"Unknown tool: {tool_name}")

    meta   = tool["_meta"]
    method = meta["method"]
    url    = meta["base_url"] + meta["path"]
    params = tool["function"]["parameters"]["properties"]

    path_params  = {}
    query_params = {}
    body         = {}

    for key, value in arguments.items():
        # Replace path placeholders like {petId}
        if f"{{{key}}}" in url:
            path_params[key] = value
        elif key in params and params[key].get("description", "").startswith("body"):
            body[key] = value
        else:
            query_params[key] = value

    # Fill path params
    for k, v in path_params.items():
        url = url.replace(f"{{{k}}}", str(v))
    print(url)
    response = requests.request(
        method  = method.upper(),
        url     = url,
        params  = query_params or None,
        json    = body or None,
        headers = {"Content-Type": "application/json"},
        timeout = 10,
    )

    return {
        "status_code": response.status_code,
        "data": response.json() if response.content else {},
    }


# ── 4. Quick smoke test ───────────────────────────────────────────────────────

result = execute_tool("getPetById", {"petId": 1}, tools)
print(result)

https://petstore.swagger.io/v2/pet/1
{'status_code': 404, 'data': {'code': 1, 'type': 'error', 'message': 'Pet not found'}}


In [6]:
from langchain_core.tools import StructuredTool

def make_lc_tool(t: dict):
    """Convert one swagger tool dict → LangChain StructuredTool."""
    fn_def = t["function"]

    # Build a pydantic model dynamically from the swagger parameters
    from pydantic import create_model
    from typing import Optional

    fields = {
        name: (Optional[str], None)   # simple typing; refine if needed
        for name in fn_def["parameters"].get("properties", {})
    }
    DynamicSchema = create_model(fn_def["name"] + "Input", **fields)

    def call_fn(**kwargs):
        result = execute_tool(fn_def["name"], kwargs, tools)  # your executor
        return json.dumps(result)

    return StructuredTool(
        name        = fn_def["name"],
        description = fn_def["description"],
        args_schema = DynamicSchema,
        func        = call_fn,
    )

swagger_lc_tools = [make_lc_tool(t) for t in tools]

In [7]:
# now bind all tools togther
from langchain.tools import tool
from pydantic import BaseModel, Field
import wikipedia
import datetime

class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")

@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""
    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            self.wiki_client.exceptions.PageError,
            self.wiki_client.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

all_tools = [search_wikipedia, get_current_temperature] + swagger_lc_tools        # ← custom + swagger together

llm_with_tools = llm.bind_tools(all_tools)
llm_with_tools

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000198BA096940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000198DB08E070>, model_name='qwen/qwen3-32b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'search_wikipedia', 'description': 'Run Wikipedia search and get page summaries.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'get_current_temperature', 'description': 'Fetch current temperature for given coordinates.', 'parameters': {'properties': {'latitude': {'description': 'Latitude of the location to fetch weather data for', 'type': 'number'}, 'longitude': {'description': 'Longitude of the location to fetch weather data for', 'type': 'number'}}, 'required': ['latitude', 'longitude'], 'type': 'object'}}}, {'type': '

In [8]:
response = llm_with_tools.invoke("tell me about pet with id 1")
response.additional_kwargs

{'reasoning_content': 'Okay, the user is asking about a pet with ID 1. Let me check the available tools. There\'s a function called getPetById which requires the petId parameter. The parameters for that function accept a string or null, so I should pass "1" as the petId. I need to make sure to format the tool call correctly within the XML tags. Let me structure the JSON with the name as "getPetById" and arguments including "petId": "1". That should retrieve the information the user is looking for.\n',
 'tool_calls': [{'id': 'tm7z58rnj',
   'function': {'arguments': '{"petId":"1"}', 'name': 'getPetById'},
   'type': 'function'}]}

In [9]:
# run this tool

function_name = response.additional_kwargs['tool_calls'][0]['function']['name']
function_parameters = json.loads(response.additional_kwargs['tool_calls'][0]['function']['arguments'])
result = execute_tool(function_name, function_parameters, tools)
result

https://petstore.swagger.io/v2/pet/1


{'status_code': 404,
 'data': {'code': 1, 'type': 'error', 'message': 'Pet not found'}}

In [10]:
llm_with_tools.invoke("What is langchain").additional_kwargs

{'reasoning_content': 'Okay, the user is asking "What is langchain." I need to figure out which tool to use here. Let me look at the available functions. There\'s a Wikipedia search function called search_wikipedia. Since the question is about defining or explaining LangChain, using that function makes sense. The other functions are related to pets, users, orders, etc., which don\'t seem relevant here. So I\'ll use search_wikipedia with the query "langchain" to get the summary from Wikipedia.\n',
 'tool_calls': [{'id': 'n6t11wsss',
   'function': {'arguments': '{"query":"langchain"}',
    'name': 'search_wikipedia'},
   'type': 'function'}]}

In [11]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])
llm_chain = prompt | llm_with_tools
llm_chain

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are helpful but sassy assistant'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000198BA096940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000198DB08E070>, model_name='qwen/qwen3-32b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'search_wikipedia', 'description': 'Run Wikipedia search and get page summaries.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type

##### Converstational Chatbot with tools

In [12]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1) create tools and llm_with_tools
tools     = all_tools
tool_map  = {t.name: t for t in tools}
llm_bound = llm.bind_tools(tools)

# 2) run the conversation with llm
def run_agent(user_input: str) -> str:
    messages = [HumanMessage(content=user_input)]

    while True:
        response = llm_bound.invoke(messages)
        messages.append(response)

        if not response.tool_calls: # if normal response return content
            return response.content

        
        for tool_call in response.tool_calls: # if tool execute it and return to AI as ToolMessage
            result = tool_map[tool_call["name"]].invoke(tool_call["args"])
            messages.append(
                ToolMessage(content=str(result), tool_call_id=tool_call["id"])
            )
            print(f"Called \'{tool_call['name']}\' with Arguments: {tool_call['args']}\nIts result is {result}")

result = run_agent("What is the weather in egypt right now?")
print(result)

Called 'get_current_temperature' with Arguments: {'latitude': 30.0444, 'longitude': 31.2357}
Its result is The current temperature is 14.1°C
The current temperature in Egypt is **14.1°C**.


In [13]:
# ReACT Agent
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver # replaces the memory

tools = all_tools # Your tools (decorated with @tool)

memory = MemorySaver() # replaces ConversationBufferMemory
agent = create_react_agent(llm, tools, checkpointer=memory) # One line replaces the entire chain + route() function

config = {"configurable": {"thread_id": "user_123"}} # thread_id is what separates conversations

# run
result = agent.invoke(
    {"messages": [("user", "What is the weather in egypt right now?")]},
    config=config # ← pass config every call
)

print(result["messages"][-1].content) # final answer

The Wikipedia search for Cairo's coordinates did not return the specific latitude and longitude needed to fetch the current temperature. To provide the weather in Egypt, I would require the exact coordinates of a city (e.g., Cairo, Alexandria). Would you like me to attempt another search or clarify the location?


In [14]:
result = agent.invoke(
    {"messages": [("user", "My name is Ahmed")]},
    config=config # ← pass config every call
)
print(result["messages"][-1].content) # final answer

Hello Ahmed! To check the current weather in Egypt, I'll need the name of a specific city (e.g., Cairo, Alexandria, or Sharm El-Sheikh). Could you clarify which location you'd like the weather for?


In [15]:
result = agent.invoke(
    {"messages": [("user", "What is my name?")]},
    config=config # ← pass config every call
)
print(result["messages"][-1].content) # final answer

Your name is **Ahmed**. You provided it earlier in the conversation. Let me know if you need further assistance! 😊


In [16]:
# Custom Visualization tool
import panel as pn
import param
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage

pn.extension()

class ChatAgent(param.Parameterized):

    def __init__(self, llm, tools, **params):
        super().__init__(**params)
        self.panels = []

        memory = MemorySaver()
        self.agent = create_react_agent(
            llm,
            tools,
            prompt=SystemMessage(content="You are helpful but sassy assistant"),
            checkpointer=memory,
        )

        # One fixed thread per chat session
        self.config = {"configurable": {"thread_id": "panel_session"}}

    def convchain(self, query):
        if not query:
            return

        inp.value = ""   # clear input box

        # ✅ Replaces AgentExecutor.invoke()
        result = self.agent.invoke(
            {"messages": [("user", query)]},
            config=self.config          # memory lives here
        )
        answer = result["messages"][-1].content

        self.panels.extend([
            pn.Row("User:",    pn.pane.Markdown(query,  width=450)),
            pn.Row("ChatBot:", pn.pane.Markdown(answer, width=450,
                               styles={"background-color": "#F6F6F6"}))
        ])
        return pn.WidgetBox(*self.panels, scroll=True)

    def clr_history(self, count=0):
        self.panels = []
        # reset thread so memory clears too
        self.config = {"configurable": {"thread_id": "panel_session_new"}}


# ── UI (unchanged from Harrison's code) ──────────────────────────────────────

cb           = ChatAgent(llm, tools)
inp          = pn.widgets.TextInput(placeholder="Enter text here…")
conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation, loading_indicator=True, height=400),
    pn.layout.Divider(),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown("# QnA_Bot")),
    pn.Tabs(("Conversation", tab1))
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'110af0ea-2eec-492c-afd5-cb12f2e39269': {'version…

#### ReAct Agent with RAG System

In [17]:
# RAG
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# a) read document and chunking
pdf_url = "https://arxiv.org/pdf/1706.03762" # Attention is All You Need
loader = PyPDFLoader(pdf_url) # downloads and reads it directly
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 50)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 93 chunks


In [18]:
# b) convert to embeddings and extract retriever
from langchain_ollama import OllamaEmbeddings # embeddings of 'lamma 3.1' or 'nomic-embed-text' or 'Dense Passage Retriever'
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [19]:
# c) add Rag system as a tool to the agent then build the agent

# retriever for pdf pages - add another rag for something else as new tool
@tool
def search_transformer_paper(query: str) -> str:
    """Search the 'Attention is All You Need' paper.
    Use for questions about attention mechanisms, transformer
    architecture, positional encoding, or training details."""

    docs = retriever.invoke(query)

    if not docs:
        return "No relevant content found in the paper."

    results = []
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page", "?")
        results.append(f"[Page {page}]:\n{doc.page_content}")

    return "\n\n---\n\n".join(results)

all_tools = [search_transformer_paper, search_wikipedia, get_current_temperature] + swagger_lc_tools # ← custom + swagger together

In [20]:
# d) build the agent (how to route is in the docstring of each tool)
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage

memory = MemorySaver()
prompt = SystemMessage(content="""
You are a helpful assistant.

Rules:
- Always cite sources and page numbers.
- Never make up information — if unsure, say so.
- If a tool fails, report the error and suggest an alternative.
- Respond in the same language the user writes in.
""")


agent = create_react_agent(
    llm,
    tools = all_tools,
    prompt = prompt,
    checkpointer = memory,
)
config = {"configurable": {"thread_id": "transformer-session"}}

def ask(question: str):
    result = agent.invoke(
        {"messages": [("user", question)]},
        config=config
    )
    print(f"Output: {result['messages'][-1].content}")

In [21]:
# Test questions
ask("What is multi-head attention?")

Output: Multi-head attention is a mechanism in the Transformer model that uses multiple parallel attention layers (called "heads") to compute different aspects of the input sequence. Here's a breakdown based on the Transformer paper:

1. **Parallel Attention Layers**:  
   The model employs $ h = 8 $ parallel attention layers (heads). Each head operates on reduced dimensionality: $ d_k = d_v = d_{\text{model}} / h = 64 $ (Page 4). This allows the model to process multiple aspects of the input simultaneously while keeping computational costs similar to single-head attention with full dimensionality.

2. **Diverse Representations**:  
   By splitting the input into multiple heads, the model can capture diverse relationships in the data. For example, different heads might focus on syntactic structures, semantic roles, or other linguistic patterns (Page 14). This specialization helps the model better understand complex dependencies in sequences.

3. **Combining Outputs**:  
   The outputs 

In [22]:
ask("What optimizer did they use for training?")

Output: The Transformer model used the **Adam optimizer** with the following parameters:  
- $ \beta_1 = 0.9 $  
- $ \beta_2 = 0.98 $  
- $ \epsilon = 10^{-9} $  

Additionally, the learning rate was adjusted dynamically during training using the formula:  
$$
\text{lrate} = d_{\text{model}}^{-0.5} \cdot \min(\text{step\_num}^{-0.5}, \text{step\_num} \cdot \text{warmup\_steps}^{-1.5})
$$  
with $ \text{warmup\_steps} = 4000 $. This schedule increases the learning rate linearly for the first 4000 steps and then decays it proportionally to the inverse square root of the step number (Page 6).  

The Adam optimizer was chosen for its efficiency in handling stochastic optimization, as noted in the paper's bibliography (Page 10 cites Kingma and Ba, 2015).


In [23]:
ask("Can you summarize what we just discussed?")

Output: Here's a summary of our discussion based on the Transformer paper:

### **Multi-Head Attention**  
- **Mechanism**: Uses $ h = 8 $ parallel attention heads to compute diverse representations of the input sequence.  
- **Dimensionality**: Each head operates on reduced dimensions ($ d_k = d_v = 64 $) to maintain computational efficiency.  
- **Purpose**:  
  - Captures different aspects of the input (e.g., syntax, semantics).  
  - Mitigates resolution loss from averaging in single-head attention.  
- **Output**: Concatenates head outputs and applies a linear projection ($ W^O $) for the final result (Page 4, Page 14).  

---

### **Optimizer and Training**  
- **Optimizer**: Adam with parameters $ \beta_1 = 0.9 $, $ \beta_2 = 0.98 $, $ \epsilon = 10^{-9} $.  
- **Learning Rate Schedule**:  
  $$
  \text{lrate} = d_{\text{model}}^{-0.5} \cdot \min(\text{step\_num}^{-0.5}, \text{step\_num} \cdot \text{warmup\_steps}^{-1.5})
  $$  
  - Warmup for 4000 steps, followed by inverse squ

In [24]:
ask("What is the weather in Cairo?")

Output: The current temperature in Cairo is **14.1°C**.


In [26]:
ask("Who is Hosni-Mubark?")

Output: Hosni Mubarak (1928–2020) was an Egyptian politician and military officer who served as **Egypt's 4th president** from **1981 to 2011**. Here's a concise summary of his life and legacy:

---

### **Key Details**  
- **Full Name**: Muhammad Hosni El Sayed Mubarak.  
- **Military Background**: A career officer in the Egyptian Air Force, he rose to the rank of Air Chief Marshal and served as its commander from 1972 to 1975.  
- **Political Career**:  
  - Vice President under Anwar Sadat (1975–1981).  
  - Assumed the presidency after Sadat's assassination in 1981.  
  - Re-elected in single-candidate referendums (1987, 1993, 1999) and later in Egypt's first multi-party election (2005).  

---

### **Presidency**  
- **Domestic Policies**:  
  - Maintained economic stability but faced criticism for widespread corruption and a repressive regime under a state of emergency (unlifted since 1967).  
  - Security forces were accused of brutality, and political dissent was heavily suppre